# Unsupervised Learning – Car Makes and Car Models with k-Means, Hierarchical Clustering and PCA

**Dataset:** Fuel consumption of 701 new car models of model year 2025 (Natural Resources Canada, Open Government
Licence – Canada): make, model, vehicle class, engine size, cylinders, transmission, fuel type and fuel consumption.
File: `../00_Data/car_fuel_consumption_2025.csv`.

**Questions:** Which car makes have a similar product range – makers of large SUVs and pickups, makers of efficient
everyday cars, premium and sports car makers? And what distinguishes one car model from another? There is **no target
variable** – we look for groups and for a compact description of the cars.

**After this exercise you can ...**
- perform a k-means iteration by hand,
- construct meaningful features for clustering,
- apply k-means (incl. scaling, elbow plot, silhouette analysis, several random starts) and interpret clusters,
- create and cut a dendrogram,
- use PCA to summarise many correlated features.

## Concept

The concepts behind this exercise (formulas and worked examples) are
explained in the notebook [clustering_pca_concept.ipynb](clustering_pca_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    adjusted_rand_score,
)
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-means by hand

*Time: about 15 minutes. Use the notebook cell below as your calculator: type the formulas with the numbers in
Python (e.g. `(3 + 5) / 2` or `0.4 ** 2`) – no paper needed.*

Six points in two dimensions: A (1, 1), B (2, 1), C (4, 3), D (5, 4), E (1, 2), F (5, 5).
We use $k = 2$ with the initial centres $\mu_1 = A = (1, 1)$ and $\mu_2 = C = (4, 3)$.

a) Compute the distances of all points to $\mu_2$ (the distances to $\mu_1$ are given) and assign every point to its
nearest centre.

b) Compute the two new centres: the mean of the $x_1$ values and the mean of the $x_2$ values of the points in each
cluster (e.g. `(1 + 2 + 1) / 3`).

c) Would any point change its cluster with the new centres? Has the algorithm converged?

In [ ]:
# Exercise 1: coordinates of the six points A, B, C, D, E, F
x1 = np.array([1, 2, 4, 5, 1, 5])
x2 = np.array([1, 1, 3, 4, 2, 5])

# a) Distances to the centres mu1 = (1, 1) and mu2 = (4, 3)
dist_1 = np.sqrt((x1 - 1) ** 2 + (x2 - 1) ** 2)
dist_2 = np.sqrt((x1 - 4) ** 2 + (x2 - 3) ** 2)
print('Distances to mu1:', dist_1.round(2))
print('Distances to mu2:', dist_2.round(2))

# b) New centres (means of the points of each cluster)
# Cluster 1 = A, B, E and cluster 2 = C, D, F
mu1 = [(1 + 2 + 1) / 3, (1 + 1 + 2) / 3]
mu2 = [(4 + 5 + 5) / 3, (3 + 4 + 5) / 3]
print('New centre 1:', np.round(mu1, 2))
print('New centre 2:', np.round(mu2, 2))

# For c): distances to the new centres - does any point change its cluster?
print('to new mu1:', np.sqrt((x1 - mu1[0]) ** 2 + (x2 - mu1[1]) ** 2).round(2))
print('to new mu2:', np.sqrt((x1 - mu2[0]) ** 2 + (x2 - mu2[1]) ** 2).round(2))

**Solution:**

a) Distances to $\mu_1=(1,1)$ / $\mu_2=(4,3)$: A 0 / 3.61, B 1 / 2.83, C 3.61 / 0, D 5 / 1.41, E 1 / 3.16, F 5.66 / 2.24
→ cluster 1 = {A, B, E}, cluster 2 = {C, D, F}.

b) $\mu_1 = \big(\tfrac{1+2+1}{3}, \tfrac{1+1+2}{3}\big) = (1.33, 1.33)$,
$\mu_2 = \big(\tfrac{4+5+5}{3}, \tfrac{3+4+5}{3}\big) = (4.67, 4)$.

c) With the new centres all points keep their cluster (A, B, E are still closer to $\mu_1$; C, D, F closer to $\mu_2$)
→ the centres do not change any more; the algorithm has converged.

## Exercise 2: Describe each car make by its product range

To cluster the car makes, we describe each make by a few features of its car models (means over all models of the make):

| feature | definition | high value means … |
|:---|:---|:---|
| `engine_size` | mean engine size [litres] | large engines |
| `consumption` | mean fuel consumption (`combined`) [L/100 km] | thirsty cars |
| `share_suv_pickup` | share of SUVs and pickups among the models | many large utility vehicles |
| `share_cvt` | share of models with a continuously variable transmission | many hybrids and efficient small cars |
| `city_highway_ratio` | mean city consumption / mean highway consumption | much more fuel in city traffic |

a) Import the car data. Compute the five features for each make (the code for the number of models and the first two
features is given). Keep only the makes with at least 5 models (with fewer models the means are unreliable).

b) Show the table sorted by `consumption` and describe the extremes.

In [ ]:
# a) Import the data as cars
cars = pd.read_csv('../00_Data/car_fuel_consumption_2025.csv', sep=',')

# Helper columns: 1 = SUV or pickup, 1 = continuously variable transmission
cars['suv_pickup'] = (
    cars['vehicle_class'].str.contains('Sport utility|Pickup').astype(int)
)
cars['cvt'] = (cars['transmission_type'] == 'continuously variable').astype(int)

# Features per make (one row per make)
makes = pd.DataFrame()
makes['models'] = cars.groupby('make')['model'].count()
makes['engine_size'] = cars.groupby('make')['engine_size'].mean()
makes['consumption'] = cars.groupby('make')['combined'].mean()
makes['share_suv_pickup'] = cars.groupby('make')['suv_pickup'].mean()
makes['share_cvt'] = cars.groupby('make')['cvt'].mean()
makes['city_highway_ratio'] = (
    cars.groupby('make')['city'].mean() / cars.groupby('make')['highway'].mean()
)

# Keep the makes with at least 5 models
makes = makes[makes['models'] >= 5]
print('Makes:', len(makes))

# b) Table sorted by consumption
print(makes.sort_values('consumption').round(2))

**Solution:**

b) 31 makes have at least 5 models. The most efficient makes are **MINI** (7.9 L/100 km), **Kia** (8.1) and Hyundai, Honda,
Mazda and Toyota (≈ 8.4–8.6): mainly small and medium engines, and at Honda and Toyota many hybrids (share of
continuously variable transmissions 57 % and 44 %). The thirstiest makes are **Ferrari** (16.5), **Rolls-Royce** (16.4) and
**INEOS** (15.8, heavy off-road vehicles), followed by GMC, Ram, Chevrolet, Porsche and Land Rover (≈ 12.4–13.2).
Rolls-Royce also has the highest city/highway ratio (1.59: in the city it uses almost 60 % more fuel than on the
highway), INEOS and Toyota the lowest (≈ 1.15). Note that the features are measured on different scales (litres,
L/100 km, shares between 0 and 1, ratios around 1.2–1.6) → standardise before clustering.

## Exercise 3: k-means – choosing k

a) Standardise the five features (not `models` – we want to group the makes by their *product range*, not by the number
of models).

b) Fit `KMeans(n_clusters=k, n_init=10, random_state=42)` for $k = 1, \dots, 8$ and plot the WCSS (`inertia_`) against
$k$ (elbow plot). Additionally compute the silhouette score for $k = 2, \dots, 8$.

c) Which $k$ do you choose?

In [ ]:
# Exercise 3: the five features
features = [
    'engine_size',
    'consumption',
    'share_suv_pickup',
    'share_cvt',
    'city_highway_ratio',
]

# a) Standardise
X = StandardScaler().fit_transform(makes[features])

# b) Elbow plot (k = 1..8) and silhouette score (k = 2..8)
ks = range(1, 9)
wcss = []
sil = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    wcss.append(km.inertia_)
    if k > 1:
        sil.append(silhouette_score(X, km.labels_))
print('WCSS:', np.round(wcss, 1))
print('Silhouette score (k = 2..8):', np.round(sil, 3))

# Figure showing the elbow plot
plt.plot(ks, wcss, marker='o')
plt.xlabel('Number of clusters k')
plt.ylabel('WCSS (inertia)')
plt.title('Elbow plot')
plt.show()

# Figure showing the silhouette score
plt.plot(range(2, 9), sil, marker='o')
plt.xlabel('Number of clusters k')
plt.ylabel('Silhouette score')
plt.show()

**Solution:**

The WCSS drops from $k=1$ (155) to $k=2$ (107), $k=3$ (73) and $k=4$ (41) and then decreases more slowly. The silhouette
score is highest for $k = 2$ (0.53) – but this "clustering" only separates Ferrari and Rolls-Royce from all other makes.
For $k = 3$ (0.37) and $k = 4$ (0.43) the scores are similar. **$k = 3$** is a sensible choice: it gives three groups of
similar size that can be interpreted well (see Exercise 4); $k = 4$ additionally puts Ferrari and Rolls-Royce into a
group of their own. With only 31 makes, the choice remains a domain decision.

## Exercise 4: Interpret the clusters

a) Fit k-means with $k = 3$ (`n_init=10, random_state=42`) and add the cluster labels to the table of the makes.

b) Compute the mean of each feature per cluster and list the makes of each cluster. Give each cluster a meaningful
name.

c) Plot the makes in a scatter plot of `engine_size` against `share_suv_pickup`, coloured by cluster, and write the name
of each make next to its point (the code for the names is given).

In [ ]:
# a) k-means with k = 3
km3 = KMeans(n_clusters=3, n_init=10, random_state=42)
km3.fit(X)
makes['cluster'] = km3.labels_

# b) Mean of each feature per cluster and makes of each cluster
print(makes.groupby('cluster')[features].mean().round(2))
for c in range(3):
    print('Cluster', c, ':', list(makes[makes['cluster'] == c].index))

# c) Figure showing the makes, coloured by cluster
plt.figure(figsize=(9, 6))
plt.scatter(makes['engine_size'], makes['share_suv_pickup'], c=makes['cluster'])

# Names of the makes next to their points
for make in makes.index:
    plt.text(
        makes.loc[make, 'engine_size'],
        makes.loc[make, 'share_suv_pickup'],
        make,
        fontsize=8,
    )
plt.xlabel('Mean engine size [litres]')
plt.ylabel('Share of SUVs and pickups')
plt.show()

**Solution:**

| cluster | typical values | name | makes |
|:---|:---|:---|:---|
| 8 makes | 96 % SUVs and pickups, ≈ 12.6 L/100 km | **SUV and pickup makes** | Chevrolet, Ford, GMC, INEOS, Jeep, Land Rover, Lincoln, Ram |
| 14 makes | smallest engines (≈ 2.3 L), ≈ 8.9 L/100 km, a third with continuously variable transmission | **efficient everyday cars** | Acura, Buick, Honda, Hyundai, Kia, Lexus, MINI, Mazda, Mitsubishi, Nissan, Subaru, Toyota, Volkswagen, Volvo |
| 9 makes | largest engines (≈ 3.8 L), ≈ 12.6 L/100 km, mostly cars instead of SUVs, high city/highway ratio (1.42) | **premium and sports cars** | Audi, BMW, Cadillac, Ferrari, Genesis, Maserati, Mercedes-Benz, Porsche, Rolls-Royce |

(The cluster numbers 0/1/2 are arbitrary.) The scatter plot confirms the interpretation: the SUV and pickup makes are at
the top (share of SUVs and pickups close to 1), the premium and sports car makes on the right (large engines) and the
efficient everyday makes on the lower left. The SUV and pickup makes and the premium makes use about the same amount of
fuel on average, but for different reasons: large, heavy vehicles vs. large, powerful engines.

## Exercise 5: Silhouette analysis

The mean silhouette score of Exercise 3 summarises a clustering in one number. A **silhouette analysis** looks at the
silhouette value $s_i = \frac{b_i - a_i}{\max(a_i, b_i)}$ of **every single make** ($a_i$ = mean distance to the
makes of its own cluster, $b_i$ = mean distance to the makes of the nearest other cluster):
$s_i$ close to 1 → well placed; close to 0 → between two clusters; negative → probably in the wrong cluster.

a) Compute the silhouette values of all makes with `silhouette_samples(X, labels)` for $k = 2, 3, 4$ and draw a
silhouette plot for each $k$: one horizontal bar per make, sorted within each cluster, with the names of the makes and a
vertical line at the mean silhouette score (the plotting code is given).

b) Which makes are badly placed for $k = 3$? Compare the three plots. Does the silhouette analysis support $k = 3$?

In [ ]:
# Function: draws one bar per make with its silhouette value, by cluster
def silhouette_plot(X, labels, names):
    """Silhouette plot: one bar per make, sorted within clusters."""
    values = silhouette_samples(X, labels)
    table = pd.DataFrame({'name': names, 'cluster': labels, 'sil': values})
    table = table.sort_values(['cluster', 'sil'])
    colors = plt.colormaps['tab10'](table['cluster'])
    plt.barh(table['name'], table['sil'], color=colors)
    plt.axvline(values.mean(), color='black', linestyle='--')
    plt.xlabel('silhouette value')
    plt.title(
        'k = '
        + str(len(np.unique(labels)))
        + ': mean = '
        + str(round(values.mean(), 2))
    )


# a) Silhouette plots for k = 2, 3 and 4 (one plot per k)
names = list(makes.index)
plt.figure(figsize=(18, 8))
for i, k in enumerate([2, 3, 4]):
    labels_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    plt.subplot(1, 3, i + 1)
    silhouette_plot(X, labels_k, names)
plt.tight_layout()
plt.show()

# b) Silhouette value of each make for k = 3
makes['silhouette'] = silhouette_samples(X, km3.labels_)
print(makes[['cluster', 'silhouette']].sort_values('cluster').round(2))
print(makes.groupby('cluster')['silhouette'].mean().round(2))

**Solution:**

a) and b) For $k = 3$ the **SUV and pickup makes** form the most compact cluster (mean silhouette ≈ 0.54; only Lincoln
is close to the border with 0.27). The other two clusters are less compact (≈ 0.30 and 0.33). Badly placed are above all
**MINI** (≈ 0.10), **Volkswagen** (≈ 0.15), Mitsubishi and Buick (≈ 0.2): they lie between the efficient everyday makes
and the premium makes (e.g. MINI and Volkswagen have small engines but no hybrids). No make has a negative value.

- $k = 2$ (mean ≈ 0.53): Ferrari and Rolls-Royce vs. all other makes – a high score, but not a useful grouping.
- $k = 4$ (mean ≈ 0.43): Ferrari and Rolls-Royce form a cluster of their own; the other makes are split into SUV and
  pickup makes, premium makes and efficient makes.

Conclusion: the silhouette analysis shows that the groups are not very sharp – many makes have a mixed product range.
$k = 3$ (or $k = 4$) are reasonable choices; a silhouette analysis is a valuable complement to the elbow plot – it shows
*which* observations are badly clustered.

## Exercise 6: Stability of k-means

k-means depends on the random initial centres.

a) Fit k-means with $k = 3$ and `n_init=1` for `random_state` = 0, 1, …, 19. Print the WCSS of each run and compare the
cluster assignment with the solution of Exercise 4 using the *adjusted Rand index* (ARI: 1 = identical partition,
independent of the cluster numbers; ≈ 0 = random agreement).

b) What do you conclude? Why is `n_init=10` a good idea?

In [ ]:
# Exercise 6: k-means with n_init=1 for random_state = 0, ..., 19;
#    WCSS and ARI compared with km3
for seed in range(20):
    km = KMeans(n_clusters=3, n_init=1, random_state=seed)
    km.fit(X)
    ari = adjusted_rand_score(km3.labels_, km.labels_)
    print(
        'random_state =',
        seed,
        ' WCSS =',
        round(km.inertia_, 2),
        ' ARI =',
        round(ari, 3),
    )
print('Reference solution (n_init=10): WCSS =', round(km3.inertia_, 2))

**Solution:**

Only 2 of the 20 runs with `n_init=1` find the reference solution (WCSS ≈ 72.8, ARI = 1). The other runs end in different
local minima with a slightly or clearly larger WCSS (≈ 72.7–83.2) and partly very different groups (ARI between ≈ 0.1
and 0.7). Interestingly, one run (random_state = 8) finds an even slightly smaller WCSS (72.68) with quite different
groups (ARI ≈ 0.11): when the groups are not sharp, several different solutions are almost equally good. k-means only
guarantees a **local** optimum. With `n_init=10` the algorithm is started 10 times and the solution with the smallest
WCSS is kept – this makes the result much more reliable (but the best of 10 starts is still not guaranteed to be the
global optimum).

## Exercise 7: Hierarchical clustering and dendrogram

a) Compute the hierarchical clustering of the standardised features with `linkage(X, method='ward')` and draw the
dendrogram with the names of the makes as labels.

b) Cut the dendrogram into 3 clusters (`fcluster(Z, t=3, criterion='maxclust')`) and compare them with the k-means
clusters (`pd.crosstab`).

c) At which height would you cut the dendrogram? Which makes are merged last?

In [ ]:
# a) Ward linkage and dendrogram
Z = linkage(X, method='ward')

# Figure showing the dendrogram of the makes
plt.figure(figsize=(12, 5))
dendrogram(Z, labels=list(makes.index), leaf_rotation=90)
plt.ylabel('Merge distance (Ward)')
plt.title('Dendrogram of the car makes')
plt.show()

# b) Cut into 3 clusters and compare with k-means
makes['hc_cluster'] = fcluster(Z, t=3, criterion='maxclust')
print(pd.crosstab(makes['cluster'], makes['hc_cluster']))
ari = adjusted_rand_score(makes['cluster'], makes['hc_cluster'])
print('ARI k-means vs. hierarchical:', round(ari, 3))
for c in [1, 2, 3]:
    print(
        'Hierarchical cluster',
        c,
        ':',
        list(makes[makes['hc_cluster'] == c].index),
    )

**Solution:**

b) The hierarchical clustering (Ward) finds **different groups** than k-means (ARI ≈ 0.34): it puts Ferrari and
Rolls-Royce into a cluster of their own, keeps 11 efficient everyday makes together and merges the SUV and pickup makes
with the premium makes (and with MINI, Volkswagen and Volvo) into one large cluster. When two methods disagree like
this, the structure in the data is not very clear – the groups depend on the method and should be interpreted with
care.

c) Ferrari and Rolls-Royce are merged with the other makes only at the very end, at a large distance – they are the most
distinct makes. A cut below the last merges gives 3 clusters. Cutting a bit lower gives 4 clusters – Ferrari and
Rolls-Royce, the efficient everyday makes, the SUV and pickup makes and the premium makes – which largely agree with
the k-means solution with $k = 4$ (ARI ≈ 0.71).

## Exercise 8: PCA – what makes car models different?

Now we turn the perspective around: the **701 car models** are the observations, five numerical features describe each
model: `engine_size`, `cylinders`, `city`, `highway` (fuel consumption in L/100 km) and `co2` (CO₂ emissions in g/km).

a) Standardise the five features and fit a PCA. Plot the explained variance ratio of the components. How much variance
does PC1 explain?

b) Show the loadings of PC1 and PC2 (`pca.components_`). Interpret both components.

c) Plot the car models in the PC1–PC2 plane, coloured by transmission (continuously variable or other). Which models
have the highest and the lowest PC2 scores?

In [ ]:
# a) The five numerical features of the 701 car models
pca_features = ['engine_size', 'cylinders', 'city', 'highway', 'co2']
Z_cars = StandardScaler().fit_transform(cars[pca_features])
pca = PCA()
scores = pca.fit_transform(Z_cars)
evr = pca.explained_variance_ratio_
print('Explained variance ratio:', evr.round(3))

# Figure showing the explained variance of the five components
plt.bar(range(1, 6), evr)
plt.xlabel('Principal component')
plt.ylabel('Share of explained variance')
plt.show()

# b) Loadings of PC1 and PC2 for each feature
loadings = pd.DataFrame(
    {'PC1': pca.components_[0], 'PC2': pca.components_[1]}, index=pca_features
)
print(loadings.round(2))

# c) Figure showing the car models in the PC1-PC2 plane
cvt = cars['cvt'] == 1
other = cars['cvt'] == 0
plt.scatter(scores[other, 0], scores[other, 1], label='other transmissions')
plt.scatter(scores[cvt, 0], scores[cvt, 1], label='continuously variable')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.show()

# Models with the highest and the lowest PC2 scores
models = cars[['make', 'model', 'engine_size', 'city', 'highway']].copy()
models['PC2'] = scores[:, 1]
print('Highest PC2:')
print(models.nlargest(5, 'PC2').round(1))
print('Lowest PC2:')
print(models.nsmallest(5, 'PC2').round(1))

**Solution:**

a) PC1 explains **86 %** of the variance, PC2 another 11 %. The five features are so strongly correlated that one number
summarises most of the information about a car model.

b) **PC1 = size and fuel consumption:** all five features have almost the same positive loading (≈ 0.43–0.47). A car
with a high PC1 score has a large engine with many cylinders and a high fuel consumption and CO₂ emission. **PC2 = engine
size compared to fuel consumption:** engine size and cylinders load positively (≈ +0.55), highway consumption and CO₂
negatively. A high PC2 score means *a large engine with relatively moderate consumption on the highway*, a low PC2 score
*a high consumption for the engine size*.

c) The highest PC2 scores have the **Rolls-Royce** models (6.7-litre V12, 12.4 L/100 km on the highway), the lowest PC2
scores the off-road versions of the **Chevrolet Colorado/Silverado** and **GMC Canyon/Sierra** (2.7-litre engine but
≈ 14–15 L/100 km on the highway – heavy four-wheel drive vehicles with off-road tyres). Cars with a continuously variable
transmission have on average higher PC2 scores. As in the concept notebook, a few principal components are enough to
describe many correlated features.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')